In [ ]:
!pip install geopandas networkx scipy shapely fiona pandas

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point, LineString
from scipy.spatial import KDTree


file_duong_path = "Duong_Giao_Thong_UTM.gpkg"
file_den_path = "Den_Giao_Thong_UTM.gpkg"

print("Đang nạp hệ thống dữ liệu không gian")
gdf_duong = gpd.read_file(file_duong_path, engine='fiona')
gdf_den = gpd.read_file(file_den_path, engine='fiona')
print(f"✅ Đã nhận diện: {len(gdf_duong)} đoạn đường và {len(gdf_den)} điểm đèn giao thông.")


G = nx.DiGraph()
for idx, row in gdf_duong.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue


    speed = float(row['Toc_do'])
    huong = int(row['Huong_di'])

    coords = list(geom.coords)

    for i in range(len(coords) - 1):

        p_sub_start = (round(coords[i][0], 5), round(coords[i][1], 5))
        p_sub_end = (round(coords[i+1][0], 5), round(coords[i+1][1], 5))


        seg_len = math.sqrt((p_sub_end[0] - p_sub_start[0])**2 + (p_sub_end[1] - p_sub_start[1])**2)


        edge_attrs = {'length': seg_len, 'speed': speed}


        if huong == 1:
            G.add_edge(p_sub_start, p_sub_end, **edge_attrs)
        elif huong == 2:
            G.add_edge(p_sub_end, p_sub_start, **edge_attrs)
        else:
            G.add_edge(p_sub_start, p_sub_end, **edge_attrs)
            G.add_edge(p_sub_end, p_sub_start, **edge_attrs)


nodes_list = list(G.nodes())
nodes_array = np.array(nodes_list)
spatial_kdtree = KDTree(nodes_array)

def find_nearest_node(x, y):
    distance, index = spatial_kdtree.query([x, y])
    return nodes_list[index]

xmin, ymin, xmax, ymax = gdf_duong.total_bounds

print("🚀 Hệ thống bắt đầu chạy mô phỏng 200 lộ trình hành trình...")

so_luong_mau = 200
mau_hop_le = 0
danh_sach_toc_do_kmh = []

while mau_hop_le < so_luong_mau:
    rand_x_A, rand_y_A = random.uniform(xmin, xmax), random.uniform(ymin, ymax)
    rand_x_B, rand_y_B = random.uniform(xmin, xmax), random.uniform(ymin, ymax)

    node_A = find_nearest_node(rand_x_A, rand_y_A)
    node_B = find_nearest_node(rand_x_B, rand_y_B)

    if node_A == node_B:
        continue

    try:
        path_nodes = nx.shortest_path(G, source=node_A, target=node_B, weight='length')
    except nx.NetworkXNoPath:
        continue

    tong_quang_duong_met = 0
    thoi_gian_nen_phut = 0

    for i in range(len(path_nodes) - 1):
        u = path_nodes[i]
        v = path_nodes[i+1]
        edge_data = G[u][v]

        edge_len = edge_data['length']
        speed = edge_data['speed']

        tong_quang_duong_met += edge_len
        thoi_gian_nen_phut += ((edge_len / 1000) / speed) * 60


    if tong_quang_duong_met < 500:
        continue


    route_geom = LineString(path_nodes)


    buffer_geom = route_geom.buffer(15)
    possible_matches_idx = gdf_den.sindex.query(buffer_geom, predicate='intersects')
    so_luong_den_do = len(possible_matches_idx)
    phat_den_do = (so_luong_den_do / 2) * (50 / 60)


    phat_goc_cua = 0
    for i in range(1, len(path_nodes) - 1):
        p_prev = path_nodes[i-1]
        p_curr = path_nodes[i]
        p_next = path_nodes[i+1]


        az1 = math.degrees(math.atan2(p_curr[0] - p_prev[0], p_curr[1] - p_prev[1])) % 360
        az2 = math.degrees(math.atan2(p_next[0] - p_curr[0], p_next[1] - p_curr[1])) % 360
        diff = az2 - az1


        if diff > 180: diff -= 360
        elif diff < -180: diff += 360

        goc = abs(diff)
        if goc >= 50:
            phat_goc_cua += (3 / 60) + (goc - 50) * (0.5 / (10 * 60))


    tong_thoi_gian_phut = thoi_gian_nen_phut + phat_den_do + phat_goc_cua
    toc_do_thuc_te = (tong_quang_duong_met / 1000) / (tong_thoi_gian_phut / 60)

    danh_sach_toc_do_kmh.append(toc_do_thuc_te)
    mau_hop_le += 1

    if mau_hop_le % 20 == 0:
        print(f"   [+] Đã mô phỏng thành công {mau_hop_le}/200 lộ trình... (Vận tốc thực tế mẫu: {toc_do_thuc_te:.2f} km/h)")


print("\n=======================================================")
print("📊 BÁO CÁO KẾT QUẢ PHÂN TÍCH MẠNG LƯỚI KHÔNG GIAN")
print("=======================================================")
van_toc_trung_binh_toan_mang = sum(danh_sach_toc_do_kmh) / len(danh_sach_toc_do_kmh)
print(f"🎯 Tổng số mẫu Monte Carlo hợp lệ thu thập: {len(danh_sach_toc_do_kmh)} tuyến đường.")
print(f"🚨 VẬN TỐC TRUNG BÌNH THỰC TẾ TOÀN MẠNG LƯỚI LÀ: {van_toc_trung_binh_toan_mang:.2f} km/h")
print("=======================================================")

Đang nạp hệ thống dữ liệu không gian
✅ Đã nhận diện: 35809 đoạn đường và 403 điểm đèn giao thông.
🚀 Hệ thống bắt đầu chạy mô phỏng 200 lộ trình hành trình...
   [+] Đã mô phỏng thành công 20/200 lộ trình... (Vận tốc thực tế mẫu: 29.46 km/h)
   [+] Đã mô phỏng thành công 40/200 lộ trình... (Vận tốc thực tế mẫu: 26.07 km/h)
   [+] Đã mô phỏng thành công 60/200 lộ trình... (Vận tốc thực tế mẫu: 28.30 km/h)
   [+] Đã mô phỏng thành công 80/200 lộ trình... (Vận tốc thực tế mẫu: 32.15 km/h)
   [+] Đã mô phỏng thành công 100/200 lộ trình... (Vận tốc thực tế mẫu: 28.68 km/h)
   [+] Đã mô phỏng thành công 120/200 lộ trình... (Vận tốc thực tế mẫu: 27.49 km/h)
   [+] Đã mô phỏng thành công 140/200 lộ trình... (Vận tốc thực tế mẫu: 26.75 km/h)
   [+] Đã mô phỏng thành công 160/200 lộ trình... (Vận tốc thực tế mẫu: 22.27 km/h)
   [+] Đã mô phỏng thành công 180/200 lộ trình... (Vận tốc thực tế mẫu: 26.17 km/h)
   [+] Đã mô phỏng thành công 200/200 lộ trình... (Vận tốc thực tế mẫu: 28.01 km/h)

📊 BÁO